# PRP Adjudication Error-Bar Analysis

This notebook analyzes `prp_adjudication_df` from `adjudicated_prp_affinities.csv`.

It provides:
- Error bars for each (Abstract, PRP) based on `Affinity_Std`
- Affinity-level plots with error bars per PRP
- Counts by affinity level per PRP
- Propagated count uncertainty using Monte Carlo sampling from row-wise affinity uncertainties

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px

In [ ]:
def affinity_level_fixed(score):
    if pd.isna(score):
        return pd.NA
    try:
        x = float(score)
    except Exception:
        return pd.NA
    if 0 <= x <= 40:
        return 'Low'
    if 40 < x <= 70:
        return 'Moderate'
    if 70 < x <= 100:
        return 'High'
    return pd.NA

In [ ]:
# Locate latest benchmark run
repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
bench_root = repo_root / 'data' / 'results' / 'affinity_benchmark'
run_dirs = sorted([p for p in bench_root.iterdir() if p.is_dir()])
if not run_dirs:
    raise FileNotFoundError(f'No benchmark runs found under {bench_root}')
run_dir = run_dirs[-1]

prp_path_adjudication = run_dir / 'adjudicated_prp_affinities.csv'
prp_adjudication_df = pd.read_csv(prp_path_adjudication)

print('Run directory:', run_dir)
print('Rows:', len(prp_adjudication_df))
display(prp_adjudication_df.head(3))

In [ ]:
# Build analysis table and error bars
analysis_df = prp_adjudication_df.copy()
analysis_df['Final_Affinity'] = pd.to_numeric(analysis_df['Final_Affinity'], errors='coerce')
analysis_df['Affinity_Mean'] = pd.to_numeric(analysis_df['Affinity_Mean'], errors='coerce')
analysis_df['Affinity_Std'] = pd.to_numeric(analysis_df['Affinity_Std'], errors='coerce').fillna(0.0)
analysis_df = analysis_df.dropna(subset=['Final_Affinity', 'PRP_Name', 'Abstract_Index']).copy()

# One-sigma interval around final affinity
analysis_df['err_low'] = (analysis_df['Final_Affinity'] - analysis_df['Affinity_Std']).clip(lower=0)
analysis_df['err_high'] = (analysis_df['Final_Affinity'] + analysis_df['Affinity_Std']).clip(upper=100)
analysis_df['Affinity_Level'] = analysis_df['Final_Affinity'].apply(affinity_level_fixed)

print('Prepared rows:', len(analysis_df))
display(analysis_df[['Abstract_Index', 'PRP_Name', 'Final_Affinity', 'Affinity_Mean', 'Affinity_Std', 'err_low', 'err_high', 'Affinity_Level']].head(10))

## Affinity + Error Bars Per PRP (Improved Views)

The single scatter is replaced with three complementary plots:

- Distribution by PRP and affinity level (box + points)
- Most uncertain (Abstract, PRP) pairs ranked by `Affinity_Std`
- Faceted error-bar view by PRP with abstract index on x-axis

In [ ]:
plot_df = analysis_df.sort_values(['PRP_Name', 'Abstract_Index']).copy()

# 1) Distribution overview: easier to compare spread and central tendency per PRP
fig = px.box(
    plot_df,
    x='PRP_Name',
    y='Final_Affinity',
    color='Affinity_Level',
    points='all',
    hover_data=['Abstract_Index', 'Affinity_Mean', 'Affinity_Std', 'err_low', 'err_high'],
    category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
    title='Final Affinity Distribution by PRP and Affinity Level'
)
fig.update_layout(xaxis_title='PRP Name', yaxis_title='Final Affinity', legend_title='Affinity Level')
fig.update_xaxes(tickangle=45)
fig.show()

# 2) Rank most uncertain pairs so review can focus on high-variance cases
top_uncertain = (
    plot_df.nlargest(40, 'Affinity_Std')
    .assign(Pair_Label=lambda d: d['Abstract_Index'].astype(str) + ' | ' + d['PRP_Name'])
    .sort_values('Affinity_Std', ascending=True)
)

fig = px.bar(
    top_uncertain,
    x='Affinity_Std',
    y='Pair_Label',
    color='Affinity_Level',
    orientation='h',
    hover_data=['Final_Affinity', 'Affinity_Mean', 'err_low', 'err_high'],
    category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
    title='Top 40 Most Uncertain (Abstract, PRP) Pairs by Affinity_Std'
)
fig.update_layout(xaxis_title='Affinity Std (error size)', yaxis_title='Abstract | PRP', legend_title='Affinity Level', height=900)
fig.show()

# 3) Faceted error bars per PRP to inspect abstract-level behavior without overplotting
fig = px.scatter(
    plot_df,
    x='Abstract_Index',
    y='Final_Affinity',
    color='Affinity_Level',
    error_y='Affinity_Std',
    facet_col='PRP_Name',
    facet_col_wrap=3,
    hover_data=['Affinity_Mean', 'Affinity_Std', 'err_low', 'err_high'],
    category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
    title='Final Affinity with Error Bars by Abstract (Faceted by PRP)'
)
fig.update_layout(yaxis_title='Final Affinity', legend_title='Affinity Level', height=900)
fig.show()

In [ ]:
# Optional: PRP-level average affinity with propagated measurement uncertainty
prp_summary = (
    analysis_df.groupby('PRP_Name', as_index=False)
    .agg(
        n=('Final_Affinity', 'count'),
        mean_final_affinity=('Final_Affinity', 'mean'),
        mean_row_std=('Affinity_Std', 'mean')
    )
)

# Approximate uncertainty on PRP mean via root-sum-square of row stds / n
prp_mean_err = (
    analysis_df.assign(var=lambda d: d['Affinity_Std'] ** 2)
    .groupby('PRP_Name', as_index=False)['var']
    .sum()
    .merge(prp_summary[['PRP_Name', 'n']], on='PRP_Name', how='left')
)
prp_mean_err['mean_error'] = np.sqrt(prp_mean_err['var']) / prp_mean_err['n']
prp_summary = prp_summary.merge(prp_mean_err[['PRP_Name', 'mean_error']], on='PRP_Name', how='left')

fig = px.bar(
    prp_summary.sort_values('mean_final_affinity', ascending=False),
    x='PRP_Name',
    y='mean_final_affinity',
    error_y='mean_error',
    title='PRP Mean Final Affinity with Propagated Error Bars'
)
fig.update_layout(xaxis_title='PRP Name', yaxis_title='Mean Final Affinity')
fig.update_xaxes(tickangle=45)
fig.show()
display(prp_summary.sort_values('mean_final_affinity', ascending=False))

## Count of Abstracts by Affinity Level Per PRP

Two outputs are shown:
- Deterministic count from `Final_Affinity` labels
- Uncertain count with propagated error bars using Monte Carlo from row-wise `Affinity_Std`

In [ ]:
# Deterministic counts
count_df = (
    analysis_df.groupby(['PRP_Name', 'Affinity_Level'], as_index=False)['Abstract_Index']
    .nunique()
    .rename(columns={'Abstract_Index': 'n_abstracts'})
)

fig = px.bar(
    count_df,
    x='PRP_Name',
    y='n_abstracts',
    color='Affinity_Level',
    barmode='group',
    category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
    title='Deterministic Abstract Counts by Affinity Level and PRP'
)
fig.update_layout(xaxis_title='PRP Name', yaxis_title='Number of Abstracts', legend_title='Affinity Level')
fig.update_xaxes(tickangle=45)
fig.show()
display(count_df.sort_values(['PRP_Name', 'Affinity_Level']))

In [ ]:
# Propagate affinity uncertainty into count uncertainty
# Assumption: each row's true affinity ~ Normal(Final_Affinity, Affinity_Std), clipped to [0, 100].
n_sim = 2000
rng = np.random.default_rng(42)

base = analysis_df[['PRP_Name', 'Final_Affinity', 'Affinity_Std']].copy()
base['Affinity_Std'] = base['Affinity_Std'].fillna(0.0).clip(lower=0.0)

sim_rows = []
for sim in range(n_sim):
    sampled = rng.normal(loc=base['Final_Affinity'].to_numpy(), scale=base['Affinity_Std'].to_numpy())
    sampled = np.clip(sampled, 0, 100)

    sampled_level = np.where(
        sampled <= 40, 'Low', np.where(sampled <= 70, 'Moderate', 'High')
    )

    tmp = pd.DataFrame({
        'PRP_Name': base['PRP_Name'].to_numpy(),
        'Affinity_Level': sampled_level
    })

    c = tmp.value_counts(['PRP_Name', 'Affinity_Level']).reset_index(name='n_abstracts')
    c['sim'] = sim
    sim_rows.append(c)

sim_counts = pd.concat(sim_rows, ignore_index=True)

count_uncertainty = (
    sim_counts.groupby(['PRP_Name', 'Affinity_Level'], as_index=False)['n_abstracts']
    .agg(mean_count='mean', std_count='std')
)

# Add deterministic count for reference
count_uncertainty = count_uncertainty.merge(
    count_df,
    on=['PRP_Name', 'Affinity_Level'],
    how='left'
)

fig = px.bar(
    count_uncertainty,
    x='PRP_Name',
    y='mean_count',
    error_y='std_count',
    color='Affinity_Level',
    barmode='group',
    category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
    title='Abstract Count by Affinity Level and PRP with Propagated Error Bars'
)
fig.update_layout(xaxis_title='PRP Name', yaxis_title='Expected Number of Abstracts', legend_title='Affinity Level')
fig.update_xaxes(tickangle=45)
fig.show()

display(count_uncertainty.sort_values(['PRP_Name', 'Affinity_Level']))

# RA Adjudication Error-Bar Analysis


In [ ]:
# Locate latest benchmark run
repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
bench_root = repo_root / 'data' / 'results' / 'affinity_benchmark'
run_dirs = sorted([p for p in bench_root.iterdir() if p.is_dir()])
if not run_dirs:
    raise FileNotFoundError(f'No benchmark runs found under {bench_root}')
run_dir = run_dirs[-1]

ra_path_adjudication = run_dir / 'adjudicated_ra_affinities.csv'
ra_adjudication_df = pd.read_csv(ra_path_adjudication)

print('Run directory:', run_dir)
print('Rows:', len(ra_adjudication_df))
display(ra_adjudication_df.head(10))

In [ ]:
# Load RA-to-PRP mapping from processed CSV (generated by scripts/prepare_data.py)
mapping_path = repo_root / 'data' / 'processed' / 'ra_grouping_ra2025.csv'
ra_grouping_df = pd.read_csv(mapping_path)

required_cols = {'RA2025', 'Grouping'}
missing_cols = required_cols - set(ra_grouping_df.columns)
if missing_cols:
    raise ValueError(f'Missing required columns in mapping file: {missing_cols}')

ra_to_prp_map = (
    ra_grouping_df[['RA2025', 'Grouping']]
    .dropna(subset=['RA2025', 'Grouping'])
    .drop_duplicates()
    .rename(columns={'RA2025': 'RA2025_ID', 'Grouping': 'PRP_Name'})
)
ra_to_prp_map['RA2025_ID'] = pd.to_numeric(ra_to_prp_map['RA2025_ID'], errors='coerce').astype('Int64')

print('Mapping file:', mapping_path)
print('Mapping rows:', len(ra_to_prp_map))
display(ra_to_prp_map.head(10))

## RA to PRP Aggregation and Affinity Counts

This section maps `RA2025_ID` to `PRP_Name`, collapses repeated `(Abstract_Index, PRP_Name)` rows by taking the maximum `LLM_Affinity`, and then plots deterministic affinity-level counts by PRP.


In [ ]:
# Map RA questions to PRP names using data/processed/ra_grouping_ra2025.csv, then collapse abstract/PRP pairs
ra_analysis_df = ra_adjudication_df.copy()
ra_analysis_df['RA2025_ID'] = pd.to_numeric(ra_analysis_df['RA2025_ID'], errors='coerce').astype('Int64')
ra_analysis_df['Final_Affinity'] = pd.to_numeric(ra_analysis_df['Final_Affinity'], errors='coerce')

ra_analysis_df = ra_analysis_df.merge(
    ra_to_prp_map,
    on='RA2025_ID',
    how='left'
 )

ra_analysis_df = ra_analysis_df.dropna(subset=['Abstract_Index', 'PRP_Name', 'Final_Affinity']).copy()

ra_analysis_df = (
    ra_analysis_df.groupby(['Abstract_Index', 'PRP_Name'], as_index=False)
    .agg(Final_Affinity=('Final_Affinity', 'max'))
)
ra_analysis_df['Affinity_Level'] = ra_analysis_df['Final_Affinity'].apply(affinity_level_fixed)

print('Prepared rows:', len(ra_analysis_df))
display(ra_analysis_df.head(10))

In [ ]:
# Deterministic counts by mapped PRP name and affinity level
ra_count_df = (
    ra_analysis_df.groupby(['PRP_Name', 'Affinity_Level'], as_index=False)['Abstract_Index']
    .nunique()
    .rename(columns={'Abstract_Index': 'n_abstracts'})
)

fig = px.bar(
    ra_count_df,
    x='PRP_Name',
    y='n_abstracts',
    color='Affinity_Level',
    barmode='group',
    category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
    title='RA-to-PRP Abstract Counts by Affinity Level and PRP'
)
fig.update_layout(xaxis_title='PRP Name', yaxis_title='Number of Abstracts', legend_title='Affinity Level')
fig.update_xaxes(tickangle=45)
fig.show()

display(ra_count_df.sort_values(['PRP_Name', 'Affinity_Level']))

In [ ]:
# Combine PRP and RA counts with a source label
combined_count_df = pd.concat(
    [
        count_df.assign(source='prp'),
        ra_count_df.assign(source='ra')
    ],
    ignore_index=True
)

# Optional: inspect merged table
display(combined_count_df.sort_values(['source', 'PRP_Name', 'Affinity_Level']))

# Plot bars
fig = px.bar(
    combined_count_df[combined_count_df['Affinity_Level'] != 'Low'],  # Focus on Moderate and High for clearer comparison
    x='PRP_Name',
    y='n_abstracts',
    facet_col='Affinity_Level',
    color='source',
    barmode='group',
    category_orders={
        'Affinity_Level': ['Moderate', 'High'],
        'source': ['prp', 'ra']
    },
    title='Abstract Counts by Affinity Level, PRP, and Source (PRP vs RA)'
)
fig.update_layout(
    xaxis_title='PRP Name',
    yaxis_title='Number of Abstracts',
    legend_title='Affinity Level'
)
fig.update_xaxes(tickangle=45)
fig.show()